In [ ]:
pip install datasets pandas tqdm wikipedia-api requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.0 MB/s eta 0:00:00


In [ ]:
import os
import json
import random
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset

In [ ]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

TARGET_PER_CATEGORY = {
    "safe":   200,
    "unsafe": 200,
    "mixed":  100,
}

HARM_CATEGORIES = {
    "weapons":       ["weapon", "explosive", "bomb", "firearm", "chemical weapon", "biological", "nuclear", "cbrn"],
    "drugs":         ["drug", "narcotic", "illegal substance", "methamphetamine", "cocaine", "synthesize", "trafficking"],
    "cybersecurity": ["malware", "hack", "exploit", "phishing", "ransomware", "sql injection", "cyber", "vulnerability"],
    "violence":      ["violence", "assault", "murder", "kill", "attack", "harm someone", "violent crime", "torture"],
}

OUTPUT_DIR = "wildguard_paired_dataset"
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [ ]:
# ─────────────────────────────────────────────
# STEP 1 — LOAD WILDGUARDMIX
# ─────────────────────────────────────────────

def load_wildguard():
    print("\n📥 Loading WildGuardMix from Hugging Face...")
    ds = load_dataset("allenai/wildguardmix", "wildguardtrain", trust_remote_code=True)
    df = ds["train"].to_pandas()
    print(f"   ✅ Loaded {len(df):,} total rows")
    print(f"   Columns: {list(df.columns)}")
    return df

In [ ]:
# ─────────────────────────────────────────────
# STEP 2 — CLASSIFY SAFETY LEVEL
# ─────────────────────────────────────────────

def classify_safety(row):
    """
    Map WildGuardMix labels to safe / unsafe / mixed.
    WildGuardMix uses: prompt_harm_label, response_harm_label, response_refusal_label
    """
    prompt_label = str(row.get("prompt_harm_label", "")).lower()
    response_label = str(row.get("response_harm_label", "")).lower()

    if prompt_label == "unharmful" and response_label in ["unharmful", "nan", ""]:
        return "safe"
    elif prompt_label == "harmful" and response_label == "harmful":
        return "unsafe"
    elif prompt_label == "harmful" and response_label in ["unharmful", "nan", ""]:
        return "mixed"   # harmful prompt but safe/refused response
    elif prompt_label == "unharmful" and response_label == "harmful":
        return "mixed"   # safe-looking prompt but harmful response
    else:
        return "mixed"

In [ ]:
# ─────────────────────────────────────────────
# STEP 3 — FILTER BY HARM CATEGORY
# ─────────────────────────────────────────────

def filter_by_category(df, category, keywords):
    """Filter rows whose prompt text contains category keywords."""
    pattern = "|".join(keywords)
    mask = df["prompt"].str.lower().str.contains(pattern, na=False, regex=True)
    filtered = df[mask].copy()
    print(f"   [{category}] Keyword match: {len(filtered):,} rows")
    return filtered

In [ ]:
# ─────────────────────────────────────────────
# STEP 4 — SAMPLE TARGET COUNTS
# ─────────────────────────────────────────────

def sample_prompts(df, category):
    """Sample safe/unsafe/mixed prompts for a category."""
    df["safety_level"] = df.apply(classify_safety, axis=1)
    samples = []

    for level, target in TARGET_PER_CATEGORY.items():
        subset = df[df["safety_level"] == level]
        n = min(len(subset), target)
        if n == 0:
            print(f"   ⚠️  No {level} samples found for [{category}]")
            continue
        sampled = subset.sample(n=n, random_state=RANDOM_SEED)
        sampled = sampled.copy()
        sampled["category"] = category
        sampled["safety_level"] = level
        samples.append(sampled)
        print(f"   [{category}] {level}: {n} samples")

    return pd.concat(samples, ignore_index=True) if samples else pd.DataFrame()

In [ ]:
# ─────────────────────────────────────────────
# STEP 5 — SAVE PROMPTS ONLY
# ─────────────────────────────────────────────

def save_outputs(df):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Keep only relevant columns
    keep_cols = [
        "category",
        "prompt",
        "safety_level",
        "prompt_harm_label",
        "response_harm_label",
    ]
    df = df[keep_cols]

    # Save full dataset
    full_path = os.path.join(OUTPUT_DIR, "prompts_only_dataset.csv")
    df.to_csv(full_path, index=False)
    print(f"\n💾 Dataset saved → {full_path}")

    # Save per-category
    for category in HARM_CATEGORIES:
        cat_df = df[df["category"] == category]
        cat_path = os.path.join(OUTPUT_DIR, f"{category}_prompts.csv")
        cat_df.to_csv(cat_path, index=False)

    # Save per-safety-level
    for level in ["safe", "unsafe", "mixed"]:
        level_df = df[df["safety_level"] == level]
        level_path = os.path.join(OUTPUT_DIR, f"{level}_prompts.csv")
        level_df.to_csv(level_path, index=False)

    # Stats
    stats = {
        "total_prompts": len(df),
        "by_category": df["category"].value_counts().to_dict(),
        "by_safety_level": df["safety_level"].value_counts().to_dict(),
    }

    with open(os.path.join(OUTPUT_DIR, "stats.json"), "w") as f:
        json.dump(stats, f, indent=2)

In [ ]:
# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    print("=" * 60)
    print("  WildGuardMix Prompt Dataset Pipeline")
    print("=" * 60)

    # Load dataset
    df = load_wildguard()

    all_samples = []

    for category, keywords in HARM_CATEGORIES.items():
        print(f"\n🔍 Processing category: [{category.upper()}]")

        # Filter by category keywords
        cat_df = filter_by_category(df, category, keywords)

        if cat_df.empty:
            print(f"   ⚠️  No data found for [{category}] — skipping")
            continue

        # Sample safe/unsafe/mixed prompts
        sampled = sample_prompts(cat_df, category)

        if sampled.empty:
            continue

        all_samples.append(sampled)

    if not all_samples:
        print("\n❌ No data collected.")
        return

    # Combine all categories
    final_df = pd.concat(all_samples, ignore_index=True)

    # Keep only relevant columns
    keep_cols = [
        "category",
        "prompt",
        "safety_level",
        "prompt_harm_label",
        "response_harm_label",
    ]
    final_df = final_df[keep_cols]

    # Save everything
    save_outputs(final_df)

    # Print summary
    print("\n" + "=" * 60)
    print("  ✅ PIPELINE COMPLETE")
    print("=" * 60)
    print(f"  Total prompts created : {len(final_df):,}")
    print(f"  Output directory      : {OUTPUT_DIR}/")

    print("\n  Breakdown by category:")
    for cat, count in final_df["category"].value_counts().items():
        print(f"    {cat:<20} {count:>5}")

    print("\n  Breakdown by safety level:")
    for level, count in final_df["safety_level"].value_counts().items():
        print(f"    {level:<20} {count:>5}")

    print("=" * 60)


if __name__ == "__main__":
    main()

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'allenai/wildguardmix' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'allenai/wildguardmix' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  WildGuardMix Prompt Dataset Pipeline

📥 Loading WildGuardMix from Hugging Face...


README.md: 0.00B [00:00, ?B/s]

train/wildguard_train.parquet:   0%|          | 0.00/53.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/86759 [00:00<?, ? examples/s]

   ✅ Loaded 86,759 total rows
   Columns: ['prompt', 'adversarial', 'response', 'prompt_harm_label', 'response_refusal_label', 'response_harm_label', 'subcategory']

🔍 Processing category: [WEAPONS]
   [weapons] Keyword match: 1,594 rows
   [weapons] safe: 200 samples
   [weapons] unsafe: 200 samples
   [weapons] mixed: 100 samples

🔍 Processing category: [DRUGS]
   [drugs] Keyword match: 1,771 rows
   [drugs] safe: 200 samples
   [drugs] unsafe: 200 samples
   [drugs] mixed: 100 samples

🔍 Processing category: [CYBERSECURITY]
   [cybersecurity] Keyword match: 6,027 rows
   [cybersecurity] safe: 200 samples
   [cybersecurity] unsafe: 200 samples
   [cybersecurity] mixed: 100 samples

🔍 Processing category: [VIOLENCE]
   [violence] Keyword match: 6,208 rows
   [violence] safe: 200 samples
   [violence] unsafe: 200 samples
   [violence] mixed: 100 samples

💾 Dataset saved → wildguard_paired_dataset/prompts_only_dataset.csv

  ✅ PIPELINE COMPLETE
  Total prompts created : 2,000
  Output d

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
!ls /content/drive/MyDrive/Colab\ Notebooks/prompt.ipynb

'/content/drive/MyDrive/Colab Notebooks/prompt.ipynb'


In [29]:
import shutil

shutil.copy(
    "/content/drive/MyDrive/Colab Notebooks/prompt.ipynb",
    "wildguard-dataset/"
)

FileNotFoundError: Directory does not exist: wildguard-dataset/